In [6]:

import numpy as np
import pandas as pd
from joblib import dump, load
from datetime import datetime
import os
from os import path
import time
import random
import ansys
from ansys.mapdl.core import launch_mapdl, launcher, Mapdl
from ansys.mapdl import reader as mapdl_reader
from shutil import copyfile
import torch
import re


In [7]:

class ClassicFuselageEnv(object):
    def __init__(self, ip, port, n_actuators=10):
        self.ip = ip
        self.port = port
        self.n_actuators = n_actuators
        # self.mapdl = launch_mapdl(ip=self.ip, port=self.port, loglevel='ERROR', override=True, cleanup_on_exit=True, nproc=2)
        self.mapdl = Mapdl(ip=self.ip, port=self.port, timeout=90)
        self.surrogate = load('surrogate_likeDu_v22.joblib').coef_
        torch.set_default_dtype(torch.float32)
         
    def reset(self, file1=None, file2=None, mode='train'):
        self.forces = np.zeros(18, dtype=np.float32) 
        
        # file = random.choice(os.listdir(folder))
        # file = input_filename
        # filepath = path.join(folder, file)
        # print("Initial shape from: ", input_filename.split('/')[-1])
        # Parse the Ansys file
        if mode == 'Train' or mode == 'Test':
            __file__ = 'FuselageActuators'
            # Select Ansys input file (randomly)
            folder = path.join(__file__, 'AnsysFiles', mode) 
            file = file2
            filepath = path.join(folder, file)
            print("Initial shape from", file.split('.')[0])
            print(filepath)
            # Parse the Ansys file
            with open(filepath, 'r') as f:
                text = f.read()
                new_text = text.split('/com,******************* SOLVE FOR LS 1 OF 1 ****************')
                self.setup_text = new_text[0]
                new_text = text.split('! *********** WB SOLVE COMMAND ***********')
                self.finish_text = new_text[1]
                f.close()

            # Load precalculated nodal positions
            folder = path.join(__file__, 'Shapes', mode) 
            file1 = file.split(".")[0] + ".npy"
            filepath = path.join(folder, file1)
            self.initPos = np.load(filepath)
            self.displacements = np.zeros((177,2))

            # Randomly select source for target positions
            folder = path.join(__file__, 'Shapes', mode) 
            # file2 = random.choice(os.listdir(folder))
            file2 = 'SolutionInputDP53.npy'
            while file1 == file2:   # make sure files are not the same
                file2 = random.choice(os.listdir(folder))
            # Load precalculated target positions
            filepath = path.join(folder, file2)
            print("Target shape from", file2.split('.')[0])
            self.targetPos = np.load(filepath)    
            self.deviations = self._get_deviation()
            self.initDev = self.deviations # for recording
        
        elif mode == 'File':
            # Set filepath to input from environment creation
            filepath = file2

            # Parse the Ansys file
            with open(filepath, 'r') as f:
                text = f.read()
                new_text = text.split('/com,******************* SOLVE FOR LS 1 OF 1 ****************')
                self.setup_text = new_text[0]
                new_text = text.split('! *********** WB SOLVE COMMAND ***********')
                self.finish_text = new_text[1]
                f.close()

            # Get target position from Ansys
            self._run_ansys()
            self.targetPos = self._get_initPos()

            # Set filepath to input from environment creation
            filepath = file1

            # Parse the Ansys file
            with open(filepath, 'r') as f:
                text = f.read()
                new_text = text.split('/com,******************* SOLVE FOR LS 1 OF 1 ****************')
                self.setup_text = new_text[0]
                new_text = text.split('! *********** WB SOLVE COMMAND ***********')
                self.finish_text = new_text[1]
                f.close()

            # Get initial position from Ansys
            self._run_ansys()
            self.initPos = self._get_initPos()
            self.displacements = self._get_displacement()
            self.deviations = self._get_deviation()
            self.initDev = self.deviations # for recording
        
        elif mode == 'Surrogate':
            # Load precalculated nodal positions
            folder = path.join(path.dirname(__file__), 'Shapes', 'Train') 
            file1 = random.choice(os.listdir(folder))
            filepath = path.join(folder, file1)
            self.initPos = np.load(filepath)
            self.displacements = np.zeros((177,2))
            # Randomly select source for target positions
            folder = path.join(path.dirname(__file__), 'Shapes', 'Train') 
            file2 = random.choice(os.listdir(folder))
            while file1 == file2:   # make sure files are not the same
                file2 = random.choice(os.listdir(folder))
            filepath = path.join(folder, file2)
            self.targetPos = np.load(filepath) 
            
        self.deviations = self._get_deviation()
        # self.deviations = self._get_deviation().astype(np.float64)
        self.error, self.maxDev, self.mse = self._get_errors()
        self.error_initial = self.error
        # self.error_init, self.maxDev = self._get_errors()
        
    def step_surrogate(self, action):
        # torch.manual_seed(0)
        # np.random.seed(0)
        angles = np.linspace(12, -192, 18)
        action = action.detach().cpu().numpy().reshape(-1)

        # n = self.n_actuators
        self.forces += action*1000 # Action space (-1,1) scaled to (-1000lb, 1000lb)
        # idx = (abs(action)).argsort()[:9-n]
        # self.forces[idx] = 0
        
        self.forces = np.array(self.forces, dtype=np.float32)
        
        self.forces_Y = self.forces*np.cos(np.deg2rad(angles))
        self.forces_Z = self.forces*np.sin(np.deg2rad(angles))
        
        # Predict deviations from surrogate model
        u = np.dot(self.surrogate, self.forces).flatten()
        self.displacements = u.reshape((-1,2))

        # Track 
        p_init = self.initPos[:,0:2].flatten()
        p_final = p_init + u
        p_target =self.targetPos[:,0:2].flatten()
        self.deviations = p_final - p_target
        # Assemble observation
        obs = self.deviations
        obs = obs.flatten()
        
        error, maxDev = self._get_errors() #rmse, max_e
        # Terminate after one time step
        return error

        
    def step(self, action): #argmax UCB
        '''
        action: select from action_sapce: it is an array \in R^(18), with (18,)
        '''
        # n = self.n_actuators
        # if type(action) == torch.Tensor():
        #     action = action.detach().numpy()
        if isinstance(action, torch.Tensor):
            action = action.detach().cpu().numpy().reshape(-1)
        else:
            action = np.asarray(action).reshape(-1)
        
        # self.forces += action*1000 # Action space (-1,1) scaled to (-1000lb, 1000lb)
        self.forces += action*1000
        
        self.forces = np.array(self.forces, dtype=np.float32)
        # # print('Input X:', self.forces)
        
        # idx = (abs(action)).argsort()[:18-n]
        
        # self.forces[idx] = 0  #eliminate minimum forces
        
        # Run the Ansys simulation with forces
        self._run_ansys()
        # Get displacements from Ansys
        self.displacements = self._get_displacement()
        u = self.displacements.flatten()
        # Track 
        p_init = self.initPos[:,0:2].flatten() #Phi
        p_final = p_init + u #Yc + Y(F)
        p_target = self.targetPos[:,0:2].flatten()
        
        self.deviations = p_final - p_target #Yc + Y(F) - Y^*
        # Assemble observation
        obs = self.deviations
        obs = obs.flatten()
        self.error, self.maxDev, self.mse = self._get_errors() #rmse, max_e
        # Terminate after one time step
        return torch.tensor(self.error, dtype=torch.float32).reshape(1,1), self.error, self.mse
        
    def _run_ansys(self):
        self.mapdl.clear()

        # 1) Rebuild model from your WB deck
        self.mapdl.input_strings(self.setup_text)

        # 2) Enable penetration on TARGE170 (in PREP7)
        self._enable_penetration_on_targets()

        # 3) Apply your SFE loads (in PREP7)
        self._set_actuator_forces(self.mapdl, self.forces)

        # 4) Make sure we are in SOLUTION before feeding the WB "solve" block
        self.mapdl.slashsolu()                     # <<< add this line
        self.mapdl.input_strings(self.finish_text) # WB's solve/post commands

        self.result = self.mapdl.result
        self.mapdl.finish()
        return self.result
    
    def _get_deviation(self):
        '''
        Calculate the distance of the current node positions from their ideal positions
        ''' 
        finalPos = self.initPos + self.displacements
        deviations = finalPos - self.targetPos[:,0:2]
        return deviations.flatten()
    
    def _stress(self, comp_name='CM_FUSELAGE_EDGE'):
        """
        从 result 接口读取最后一组结果的节点应力；在组件 comp_name 上筛选；
        返回 node_id / 3x3 应力张量 / 各分量 / von Mises。
        """
        import numpy as np

        # 选择组件中的节点
        if comp_name:
            self.mapdl.cmsel(name=comp_name)
        sel_ids = np.asarray(self.mapdl.mesh.nnum, dtype=int).ravel()

        # 结果对象与最后一个结果集索引
        res = getattr(self, "result", None) or self.mapdl.result
        try:
            rnum = res.nsets - 1
        except Exception:
            rnum = -1

        raw = res.nodal_stress(rnum)  # 可能是 ndarray 或 tuple

        # --- 统一解析为: stress_arr [N,6], nnum_all [N] ---
        stress_arr, nnum_all = None, None

        def _as_arr(x):
            a = np.asarray(x)
            return a

        if isinstance(raw, tuple):
            # 在 tuple 里找 [N,6或7] 的那个作为应力数组；再找 [N] 的作为节点号
            for item in raw:
                a = _as_arr(item)
                if a.ndim == 2 and a.shape[0] > 0 and a.shape[1] in (6, 7):
                    stress_arr = a[:, :6]  # 前6列按 [SX,SY,SZ,SXY,SYZ,SXZ]
            for item in raw:
                a = _as_arr(item)
                if a.ndim == 1 and a.size == (stress_arr.shape[0] if stress_arr is not None else a.size):
                    # 认为是节点号（尽量取整数化）
                    nnum_all = a.astype(int)
        else:
            a = _as_arr(raw)
            if a.ndim == 2 and a.shape[1] >= 6:
                stress_arr = a[:, :6]
            # 尝试从结果网格拿节点号
            try:
                nnum_all = res.mesh.nnum.astype(int).ravel()
            except Exception:
                nnum_all = np.asarray(self.mapdl.mesh.nnum, dtype=int).ravel()

        if stress_arr is None:
            self.mapdl.allsel()
            raise RuntimeError("无法解析 nodal_stress 返回的应力数组；请打印 raw 检查其结构。")
        if nnum_all is None or nnum_all.shape[0] != stress_arr.shape[0]:
            # 再次兜底：如果尺寸不匹配，尝试用 result.mesh.nnum
            try:
                nnum_all = res.mesh.nnum.astype(int).ravel()
            except Exception:
                nnum_all = np.asarray(self.mapdl.mesh.nnum, dtype=int).ravel()
            if nnum_all.shape[0] != stress_arr.shape[0]:
                self.mapdl.allsel()
                raise RuntimeError(
                    f"节点号数量({nnum_all.shape[0]})与应力行数({stress_arr.shape[0]})不一致；"
                    "可能是选择集或结果集不匹配。"
                )

        # 根据选择集筛选
        idx_map = {nid: i for i, nid in enumerate(nnum_all)}
        pick_idx = [idx_map[nid] for nid in sel_ids if nid in idx_map]
        if not pick_idx:
            self.mapdl.allsel()
            raise RuntimeError("选定组件内没有匹配到结果中的节点。请确认组件与结果集。")

        stress = stress_arr[pick_idx, :]                # [N,6]
        node_id = sel_ids[np.isin(sel_ids, nnum_all)]   # [N]

        # 组装张量与 von Mises
        sx, sy, sz, sxy, syz, sxz = [stress[:, i] for i in range(6)]
        n = stress.shape[0]
        sigma = np.zeros((n, 3, 3), dtype=float)
        sigma[:, 0, 0] = sx
        sigma[:, 1, 1] = sy
        sigma[:, 2, 2] = sz
        sigma[:, 0, 1] = sigma[:, 1, 0] = sxy
        sigma[:, 1, 2] = sigma[:, 2, 1] = syz
        sigma[:, 0, 2] = sigma[:, 2, 0] = sxz

        von_mises = np.sqrt(
            0.5 * ((sx - sy) ** 2 + (sy - sz) ** 2 + (sz - sx) ** 2) +
            3.0 * (sxy ** 2 + syz ** 2 + sxz ** 2)
        )

        self.mapdl.allsel()
        return {
            "node_id": node_id,
            "sigma_tensor": sigma,
            "components": {
                "SX": sx, "SY": sy, "SZ": sz,
                "SXY": sxy, "SYZ": syz, "SXZ": sxz,
            },
            "von_mises": von_mises,
        }
        
    
    def _get_displacement(self):
        '''
        Get the displacement of nodes on the fuselage edge after forces have been applied. 
        The displacements are relative to the initial positions of the nodes.
        '''
        self.mapdl.cmsel(name='CM_FUSELAGE_EDGE') # select nodes on the edge of the fuselage
        displacements = self.mapdl.post_processing.nodal_displacement('ALL') # get displacements of nodes on the edge
        self.mapdl.allsel()
        return displacements[:,1:3]
            
    def _get_initPos(self):
        '''
        Get the initial positions of nodes on the fuselage edge before any forces are applied
        '''
        self.mapdl.cmsel(name='CM_FUSELAGE_EDGE') # select nodes on the edge of the fuselage
        initPos = self.mapdl.mesh.nodes # initial positions of nodes
        nnum = self.mapdl.mesh.nnum # corresponding node numbers
        self.mapdl.allsel()
        return initPos[:,1:3]
        
    def _get_obs(self):
        # Get displacements from simulation
        self.displacements = self._get_displacement()
        # Calculate deviations
        self.deviations = self._get_deviation()
        obs = self.deviations
        obs = obs.flatten() #np.expand_dims(obs, -1)
        return np.array(obs, dtype=np.float32)

    def _get_errors(self):
        # Needs to be called after getting observations so that data is up to date
        # Calculate error relative to perfect circle with r=288
        n = len(self.deviations)
        dev_total = np.sqrt(np.square(self.deviations[:177]) + np.square(self.deviations[177:]))
        max_e = max(dev_total) # maximum error
        mae = sum(np.abs(self.deviations))/n # mean absolute error
        rmse = np.sqrt(sum((self.deviations)**2)/n) # root mean squared error
        mse = sum((self.deviations)**2)/n # mean squared error
        se = sum((self.deviations)**2) # sum of squared errors
        return mae, max_e, mse

    def _record(self):
        # Build dataframes
        df1 = pd.DataFrame(self.initDev, self.h1).T
        df2 = pd.DataFrame(self.forces, self.h2).T
        df3 = pd.DataFrame(self.finalDev, self.h3).T
        # Join them together
        df = pd.concat([df1, df2, df3], axis=1)
        # Write csv file
        df.to_csv(self.recordPath, mode='a', header=not os.path.exists(self.recordPath))

    def _launch_ansys(self):
        self.mapdl = Mapdl(ip=self.ip, port=8800)

    def _set_actuator_forces(self, mapdl, forces):
        # Enter PREP7 so SFE is valid
        mapdl.prep7()

        # Calculate y and z components of the forces
        angles = np.linspace(12, -192, 18)
        self.forces_Y = forces * np.cos(np.deg2rad(angles))
        self.forces_Z = forces * np.sin(np.deg2rad(angles))

        # Apply the forces as surface pressure on selected elements by REAL set
        for i in range(18):
            # X component (tiny)
            mapdl.esel("S", "REAL", "", 27 + 3*i)
            mapdl.sfe("ALL", 1, "PRES", 1, 2.24808943074769e-009)

            # Y component
            mapdl.esel("S", "REAL", "", 28 + 3*i)
            mapdl.sfe("ALL", 1, "PRES", 1, float(self.forces_Y[i]))

            # Z component
            mapdl.esel("S", "REAL", "", 29 + 3*i)
            mapdl.sfe("ALL", 1, "PRES", 1, float(self.forces_Z[i]))

        mapdl.esel("ALL")   # restore selection
        mapdl.finish()      # leave PREP7
        
    def _enable_penetration_on_targets(self):
        """Enable fluid penetration on ALL TARGE170 types in the model."""
        # Must be in PREP7 for KEYOPT to work
        self.mapdl.prep7()

        # List element types and parse the TARGE170 type numbers
        et_text = self.mapdl.etlist()     # returns the ETLIST text
        targe_types = [int(m.group(1)) for m in re.finditer(
            r"ELEMENT TYPE\s+(\d+)\s+IS\s+TARGE170", et_text)]

        # Flip KEYOPT(10)=1 on every TARGE170 type we found
        for etnum in targe_types:
            self.mapdl.keyopt(etnum, 10, 1)

        self.mapdl.finish()
        
        # 放进 ClassicFuselageEnv 类体内 ---------------------------------------------
    def plot_edge_matplotlib(self, scale=1.0, show_init=True, show_target=True, show_deformed=True,
                            figsize=(6,6), save=None, title=None):
        """
        用 matplotlib 画机身边缘节点的 2D 形状（Y–Z 或你当前存的两列）。
        假设:
        - self.initPos:  [N, >=2]，reset 时已载入
        - self.targetPos:[N, >=2]，reset 时已载入
        - self.displacements: [N, 2]，step()/step_surrogate() 或 _get_displacement() 后更新

        参数
        ----
        scale : float
            变形放大倍数（对 self.displacements 生效）。
        show_init / show_target / show_deformed : bool
            控制显示哪些曲线。
        figsize : tuple
            图像尺寸。
        save : str or None
            文件名（如 'fuselage.png'），不为 None 则保存。
        title : str or None
            标题。
        """
        import numpy as np
        import matplotlib.pyplot as plt

        # 取两列坐标（你上游代码用的是 [:,0:2]）
        P_init = np.asarray(self.initPos)[:, 0:2]
        P_tgt  = np.asarray(self.targetPos)[:, 0:2]

        # 若没有位移，给零
        if getattr(self, "displacements", None) is None or self.displacements is None:
            disp = np.zeros_like(P_init)
        else:
            disp = np.asarray(self.displacements)

        # 变形后位置
        P_def = P_init + scale * disp

        # 尝试按极角排序，让折线更“顺滑”（可选）
        # 以质心为参考，按 atan2 排序
        center = P_init.mean(axis=0)
        def sort_by_angle(P):
            ang = np.arctan2(P[:,1] - center[1], P[:,0] - center[0])
            idx = np.argsort(ang)
            return P[idx]

        P_init_s = sort_by_angle(P_init)
        P_tgt_s  = sort_by_angle(P_tgt)
        P_def_s  = sort_by_angle(P_def)

        plt.figure(figsize=figsize)
        if show_init:
            plt.plot(P_init_s[:,0], P_init_s[:,1], '-', lw=1.5, label='Initial')
        if show_target:
            plt.plot(P_tgt_s[:,0], P_tgt_s[:,1], '--', lw=1.5, label='Target')
        if show_deformed:
            plt.plot(P_def_s[:,0], P_def_s[:,1], '-', lw=2.0, label=f'Deformed (x{scale:g})')

        plt.gca().set_aspect('equal', adjustable='box')
        plt.xlabel('Axis-1')
        plt.ylabel('Axis-2')
        plt.grid(True, alpha=0.3)
        if title:
            plt.title(title)
        plt.legend(loc='best')
        plt.tight_layout()

        if save:
            plt.savefig(save, dpi=300)
        plt.show()

    def plot_single_layer(self, layer_index=1, comp_name='CM_FUSELAGE_EDGE',
                        figsize=(6,6), save=None, title=None):
        """
        绘制指定层 (SHELL181 层合板的 layer_index) 的节点投影形状。
        layer_index 从 1 开始计数。
        """
        import numpy as np
        import matplotlib.pyplot as plt

        # 选择壳体边缘节点
        if comp_name:
            self.mapdl.cmsel(name=comp_name)

        # 拿到当前选择的节点坐标
        nodes = np.array(self.mapdl.mesh.nodes)
        if nodes.ndim != 2 or nodes.shape[1] < 4:
            raise RuntimeError("mesh.nodes 格式不正确，期望至少4列 [id,x,y,z]。")
        xyz = nodes[:, 1:4]
        self.mapdl.allsel()

        # 读取指定层的厚度信息（如果可取）
        try:
            res = getattr(self, "result", None) or self.mapdl.result
            layer_data = res.shell_layer_data(layer_index)  # [n_elem, n_layers, 6] 或类似格式
            print(f"Layer {layer_index} data shape:", np.shape(layer_data))
        except Exception:
            layer_data = None

        # 按投影画 2D（假设 y-z 面）
        plt.figure(figsize=figsize)
        plt.plot(xyz[:,1], xyz[:,2], 'o-', lw=1.5, ms=3, label=f'Layer {layer_index}')
        plt.gca().set_aspect('equal', adjustable='box')
        plt.xlabel('Y'); plt.ylabel('Z')
        plt.title(title or f"Fuselage Layer {layer_index}")
        plt.grid(True, alpha=0.3)
        plt.legend()
        if save:
            plt.savefig(save, dpi=300)
        plt.show()

In [8]:
torch.set_default_dtype(torch.float32)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
timestamp = datetime.now().strftime('%Y%m%d%H%M%S')
seed = 2
random.seed(seed)
bounds = torch.tensor([[-1.0] * 18, [1.0] * 18], device=device)
###
__file__ = 'FuselageActuators'
folder = path.join(__file__, 'AnsysFiles', "Test") 
# file = random.choice(os.listdir(folder))
file = 'SolutionInputDP52.inp'
filepath = path.join(folder, file)
original_input_filename = filepath
print("Initial shape from", file.split('.')[0])
###

log_dir = "Experiments"
if not os.path.exists(log_dir):
        os.makedirs(log_dir)
        
env_name = "Quantum_EXP_10_final"
log_dir = log_dir + '/' + env_name + '/' + 'exp_set_0/'
if not os.path.exists(log_dir):
        os.makedirs(log_dir)

input_filename = log_dir + file
copyfile(original_input_filename, input_filename)



file1 = './FuselageActuators/AnsysFiles/Benchmark/SolutionInputUndeformed.inp'
file2 = './FuselageActuators/AnsysFiles/Benchmark/SolutionInputUndeformed.inp'

perfectPos= pd.read_csv("perfectPos.csv").values


Initial shape from SolutionInputDP52


In [ ]:
import os
from os import path
import numpy as np

import torch
import matplotlib.pyplot as plt

from quantum_bo_env import QuantumFuselageEnv

def load_data_wo_constraint(filename, device):
    data = torch.load(filename)
    actions = data['actions'].to(torch.float32).to(device)
    true_response = data['true_response'].to(torch.float32).to(device)
    response = data['response'].to(torch.float32).to(device)
    num_of_queries =  data['queries'].to(device)
    eps = data['uncertainty']
    error_init = data['error_init']
    return actions, true_response, response, num_of_queries, eps, error_init

print(input_filename)

obs_noise = 0.1**2
env =  ClassicFuselageEnv(ip="129.161.90.56", port=8800) 
  
env.reset(file2=input_filename.split('/')[-1], mode='Test')

# actions, true_response, response, num_of_queries, eps, error_init = load_data_wo_constraint('Experiments/Quantum_EXP_10_1_0.5_2/exp_set_0/' + str(obs_noise)+ 'quan_training_data_0_.pth', device='cuda:0')
actions, true_response, response, num_of_queries, eps, error_init = load_data_wo_constraint('Experiments/Classic_EXP_10_1_0.5_1/exp_set_0/' + str(obs_noise)+ 'training_data_0_.pth', device='cuda:0')


Experiments/Quantum_EXP_10_final/exp_set_0/SolutionInputDP52.inp


MapdlConnectionError: Unable to connect to MAPDL gRPC instance at dns:///129.161.91.254:8800.
Reached either maximum amount of connection attempts (5) or timeout (90 s).The MAPDL process has died.

In [ ]:
force_actions = np.asarray([ 0.        ,  0.        , -0.06094483, -0.05967504,  0.19094917,
  0.10664172, -0.3558838 ,  0.19167042, -0.04168794, -0.13555796,
 -0.09844309,  0.        ,  0.        ,  0.        ,  0.        ,
 -0.2363561 ,  0.        ,  0.        ])

# n = 10
# idx = (abs(force_actions)).argsort()[:18-n]
# force_actions[idx] = 0

print(force_actions)
# action3 = torch.tensor([-0.2006, -0.0770,  0.3100, -0.4840,  0.0000,  0.0000,  0.0000,  0.0000,
#          0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.1346,  0.3932,
#         -0.4649, -0.4861], device='cuda:0')

final_error_tensor, final_error, mse = env.step(force_actions)
finalPos = env.initPos + env.displacements # envs.get_attr("initPos")[0] + envs.get_attr("displacements")[0]
finalDev = env.deviations #get_attr("deviations")[0]
forces = env.forces #get_attr("forces")[0]
forcesY = env.forces_Y #get_attr("forces_Y")[0]
forcesZ = env.forces_Z #get_attr("forces_Z")[0]
forcesActive = np.argwhere(forces!=0)
forcesInactive = np.argwhere(forces==0)


initPos = env.initPos
targetPos = env.targetPos
initError = env.error_initial
finalPos = env.initPos + env.displacements

# Initial and final positions for visualization
mag = 25
initPosVis = perfectPos + (initPos - perfectPos)*mag
targetPosVis = perfectPos + (targetPos - perfectPos)*mag 
finalPosVis = targetPos + (finalPos-perfectPos)*mag

angles = np.linspace(12, -192, 18)
angles[17] = 168

anglesTarget = np.rad2deg(np.arctan2(env.targetPos[:,1],env.targetPos[:,0]))
anglesInit = np.rad2deg(np.arctan2(initPos[:,1],initPos[:,0]))

actuatorIds = np.absolute(np.expand_dims(angles,1)-np.expand_dims(anglesInit,1).T).argmin(axis=1)

# Plot
fig, axs = plt.subplots(
    nrows=1, ncols=2,
    figsize=(11, 5),   # 控制物理尺寸（英寸）
    dpi=600            # 🔴 提高分辨率，论文一般 300 或 600
)
fig.suptitle('Fuselage Edge Deviations (magnified x%i)' %(mag, ))
# # Plot of initial positions
axs[0].plot(targetPosVis[:,0], targetPosVis[:,1], '.')
axs[0].plot(initPosVis[:,0], initPosVis[:,1], '.')
# # axs[0].plot(targetPos[actuatorIds,0], targetPos[actuatorIds,1], 'x')
# # C= np.sqrt(initDev[0]**2+initDev[1]**2) # magnitude of displacements - use for color in quivers
# # axs[0].quiver(initPosVis[:,0], initPosVis[:,1], targetPos[:,0]-initPosVis[:,0], targetPos[:,1]-initPosVis[:,1], C, angles='xy', scale=1, units='xy')
axs[0].axes.set_aspect('equal')
axs[0].set_xlabel('Y [in]')
axs[0].set_ylabel('Z [in]')
axs[0].set_title('Initial Shape (MAE = %.3f in)' %initError)
axs[0].legend(['Target shape', 'Initial shape'], loc='center')
axs[0].set_xlim(-170, 170)
axs[0].set_ylim(-170, 170)

# # # Plot of final positions
axs[1].plot(targetPosVis[:,0], targetPosVis[:,1], '.')
axs[1].plot(finalPosVis[:,0], finalPosVis[:,1], '.')
# # # C= np.sqrt(finalDev) # magnitude of displacements - use for color in quivers


# axs[1].quiver(finalPosVis[:,0], finalPosVis[:,1], targetPos[:,0]-finalPosVis[:,0], targetPos[:,1]-finalPosVis[:,1], C, angles='xy', scale=1, units='xy')
axs[1].plot(finalPosVis[actuatorIds[forcesInactive],0], finalPosVis[actuatorIds[forcesInactive],1], 'x', color='grey', markersize=10, markeredgewidth=2)
axs[1].plot(finalPosVis[actuatorIds[forcesActive],0], finalPosVis[actuatorIds[forcesActive],1], '*', color='cyan', markersize=6)
axs[1].quiver(finalPosVis[actuatorIds[forcesActive],0], finalPosVis[actuatorIds[forcesActive],1], forcesY[forcesActive], forcesZ[forcesActive], angles='xy', scale=15, units='xy')

axs[1].axes.set_aspect('equal')
axs[1].set_xlabel('Y [in]')
axs[1].set_ylabel('Z [in]')
axs[1].set_title('Adjusted Shape (MSE = %.3f in)' %mse.item())
first_legend = axs[1].legend(['Target shape', 'Adjusted shape'], loc='center')


# # # Add the legend manually to the current Axes.
axs[1].add_artist(first_legend)
# # # Create another legend for the second line.
axs[1].legend(['_nolegend_','_nolegend_','Unused actuators', 'Selected actuators', 'Force vectors'], loc='lower right', bbox_to_anchor=(1.55, 0.0))

axs[1].set_xlim(-170, 170)
axs[1].set_ylim(-170, 170)

plt.draw()

In [ ]:
min_idx2 = torch.argmin(true_response[:415]).item()
actions2 = actions[min_idx2, :]
min_idx1 = torch.argmin(true_response[:100]).item()
actions1 = actions[min_idx1, :]
min_idx3 = torch.argmin(true_response).item()
# print(response[min_idx])
actions3 = actions[min_idx3, :]

In [ ]:
actions3

In [ ]:
# min_idx2 = torch.argmin(true_response[:500]).item()
# actions2 = actions[min_idx2, :]
# min_idx1 = torch.argmin(true_response[:100]).item()
# actions1 = actions[min_idx1, :]
# min_idx3 = torch.argmin(true_response).item()
# # print(response[min_idx])
# actions3 = actions[min_idx3, :]

In [ ]:
action_list = [actions1, actions2, actions3]

In [ ]:
# --- Step 0: original eval for init/target, no actions needed ---
initPos = env.initPos
targetPos = env.targetPos
initError = env.error_initial

mag = 25
initPosVis   = perfectPos + (initPos   - perfectPos) * mag
targetPosVis = perfectPos + (targetPos - perfectPos) * mag

angles = np.linspace(12, -192, 18)
angles[17] = 168

anglesTarget = np.rad2deg(np.arctan2(env.targetPos[:,1], env.targetPos[:,0]))
anglesInit   = np.rad2deg(np.arctan2(initPos[:,1],       initPos[:,0]))
actuatorIds  = np.absolute(np.expand_dims(angles,1) - np.expand_dims(anglesInit,1).T).argmin(axis=1)

# --- Step 1: make a 1x4 grid ---
fig, axs = plt.subplots(
    nrows=1, ncols=4,
    figsize=(18, 4),
    dpi=600
)
fig.suptitle('Classic Bayesian Optimization Adjustment Results: Fuselage Edge Deviations (magnified x%i)' % mag)

# --- Panel 0: original shape (no action) ---
ax0 = axs[0]
ax0.plot(targetPosVis[:,0], targetPosVis[:,1], '.')
ax0.plot(initPosVis[:,0],   initPosVis[:,1],   '.')
ax0.set_aspect('equal')
ax0.set_xlabel('Y [in]')
ax0.set_ylabel('Z [in]')
ax0.set_title('Initial Shape (MAE = %.3f in)' % initError)
ax0.legend(['Target shape', 'Initial shape'], loc='center')
ax0.set_xlim(-170, 170)
ax0.set_ylim(-170, 170)

for i in range(3):
    # 👉 为当前 action 单独创建一个 env_i，避免覆盖外面的 env
    env_i = ClassicFuselageEnv(ip='128.213.75.123', grpc_port=8800)
    env_i.reset(file2=input_filename.split('/')[-1], mode='Test')

    # 2) apply the i-th action
    action_i = action_list[i]  # shape (n_actuators,)
    # 如果 action_i 是 torch.Tensor，可以加一句：
    # action_i = action_i.detach().cpu().numpy()

    final_error_tensor, final_error = env_i.step(action_i)

    # 3) pull data from env_i for this action
    finalPos = env_i.initPos + env_i.displacements
    finalDev = env_i.deviations
    forces   = np.asarray(env_i.forces,   dtype=float).flatten()
    forcesY  = np.asarray(env_i.forces_Y, dtype=float).flatten()
    forcesZ  = np.asarray(env_i.forces_Z, dtype=float).flatten()

    # 这里用 where 更直观一点
    forcesActive   = np.where(forces != 0)[0]
    forcesInactive = np.where(forces == 0)[0]

    # 4) visualize positions for this action（注意：仍然用外面统一的 perfectPos / targetPos）
    finalPosVis = targetPos + (finalPos - perfectPos) * mag

    ax = axs[i + 1]  # panels 1, 2, 3

    # target & adjusted shapes
    ax.plot(targetPosVis[:, 0], targetPosVis[:, 1], '.')
    ax.plot(finalPosVis[:, 0],  finalPosVis[:, 1],  '.')

    # unused vs used actuators
    ax.plot(
        finalPosVis[actuatorIds[forcesInactive], 0],
        finalPosVis[actuatorIds[forcesInactive], 1],
        'x', color='grey', markersize=8, markeredgewidth=1.5
    )
    ax.plot(
        finalPosVis[actuatorIds[forcesActive], 0],
        finalPosVis[actuatorIds[forcesActive], 1],
        '*', color='cyan', markersize=6
    )

    # force vectors (quiver)
    ax.quiver(
        finalPosVis[actuatorIds[forcesActive], 0],
        finalPosVis[actuatorIds[forcesActive], 1],
        forcesY[forcesActive],
        forcesZ[forcesActive],
        angles='xy', scale=15, units='xy'
    )

    ax.set_aspect('equal')
    ax.set_xlabel('Y [in]')
    # if i == 0:   # 只在第一个子图列加 ylabel，避免太挤
    #     ax.set_ylabel('Z [in]')
    ax.set_title('Action %d (MAE = %.3f in)' % (i + 1, final_error.item()))

    # 只在最后一个子图上放完整 legend
    # if i == 2:
    #     first_legend = ax.legend(['Target shape', 'Adjusted shape'], loc='center')
    #     ax.add_artist(first_legend)
    #     ax.legend(
    #         ['_nolegend_', '_nolegend_', 'Unused actuators', 'Selected actuators', 'Force vectors'],
    #         loc='lower right', bbox_to_anchor=(1.05, 0.0)
    #     )

    if i == 2:
        ax.legend(
            ['Unused actuators', 'Selected actuators', 'Force vectors'],
            loc='upper left',
            bbox_to_anchor=(1.05, 1),   # 👉 往右挪 + 往上挪
            borderaxespad=0,
            frameon=True
        )

    ax.set_xlim(-170, 170)
    ax.set_ylim(-170, 170)

plt.draw()

In [ ]:
force_actions = np.asarray([-5.26813208e-07 ,-6.72385321e-07  ,2.14534148e+00  ,4.06497594e+00,\
 -5.61531514e-07, -3.89028385e-07 ,-2.10106255e-07 ,-6.86139071e-08,
  7.45639556e-09  ,1.03601235e-08 ,-5.18386269e-08, -1.60154481e-07,
 -2.85495964e-07 ,-3.90529446e-07  ,4.74188906e+00,  2.98601231e+01,
  7.99113278e+01  ,7.12587998e+01])

final_error_tensor, final_error = env.step(force_actions)
finalPos = env.initPos + env.displacements # envs.get_attr("initPos")[0] + envs.get_attr("displacements")[0]
finalDev = env.deviations #get_attr("deviations")[0]
forces = env.forces #get_attr("forces")[0]
forcesY = env.forces_Y #get_attr("forces_Y")[0]
forcesZ = env.forces_Z #get_attr("forces_Z")[0]
forcesActive = np.argwhere(forces!=0)
forcesInactive = np.argwhere(forces==0)


initPos = env.initPos
targetPos = env.targetPos
initError = env.error_initial
finalPos = env.initPos + env.displacements

# Initial and final positions for visualization
mag = 25

initPosVis = perfectPos + (initPos - perfectPos)*mag
targetPosVis = perfectPos + (targetPos - perfectPos)*mag 
finalPosVis = targetPos + (finalPos-perfectPos)*mag

angles = np.linspace(12, -192, 18)
angles[17] = 168

anglesTarget = np.rad2deg(np.arctan2(env.targetPos[:,1],env.targetPos[:,0]))
anglesInit = np.rad2deg(np.arctan2(initPos[:,1],initPos[:,0]))

actuatorIds = np.absolute(np.expand_dims(angles,1)-np.expand_dims(anglesInit,1).T).argmin(axis=1)

# Plot
fig, axs = plt.subplots(
    nrows=1, ncols=2,
    figsize=(11, 5),   # 控制物理尺寸（英寸）
    dpi=600            # 🔴 提高分辨率，论文一般 300 或 600
)
fig.suptitle('Fuselage Edge Deviations (magnified x%i)' %(mag, ))
# # Plot of initial positions
axs[0].plot(targetPosVis[:,0], targetPosVis[:,1], '.')
axs[0].plot(initPosVis[:,0], initPosVis[:,1], '.')
# # axs[0].plot(targetPos[actuatorIds,0], targetPos[actuatorIds,1], 'x')
# # C= np.sqrt(initDev[0]**2+initDev[1]**2) # magnitude of displacements - use for color in quivers
# # axs[0].quiver(initPosVis[:,0], initPosVis[:,1], targetPos[:,0]-initPosVis[:,0], targetPos[:,1]-initPosVis[:,1], C, angles='xy', scale=1, units='xy')
axs[0].axes.set_aspect('equal')
axs[0].set_xlabel('Y [in]')
axs[0].set_ylabel('Z [in]')
axs[0].set_title('Initial Shape (MAE = %.3f in)' %initError)
axs[0].legend(['Target shape', 'Initial shape'], loc='center')
axs[0].set_xlim(-170, 170)
axs[0].set_ylim(-170, 170)

# # # Plot of final positions
axs[1].plot(targetPosVis[:,0], targetPosVis[:,1], '.')
axs[1].plot(finalPosVis[:,0], finalPosVis[:,1], '.')
# # # C= np.sqrt(finalDev) # magnitude of displacements - use for color in quivers


# axs[1].quiver(finalPosVis[:,0], finalPosVis[:,1], targetPos[:,0]-finalPosVis[:,0], targetPos[:,1]-finalPosVis[:,1], C, angles='xy', scale=1, units='xy')
axs[1].plot(finalPosVis[actuatorIds[forcesInactive],0], finalPosVis[actuatorIds[forcesInactive],1], 'x', color='grey', markersize=10, markeredgewidth=2)
axs[1].plot(finalPosVis[actuatorIds[forcesActive],0], finalPosVis[actuatorIds[forcesActive],1], '*', color='cyan', markersize=6)
axs[1].quiver(finalPosVis[actuatorIds[forcesActive],0], finalPosVis[actuatorIds[forcesActive],1], forcesY[forcesActive], forcesZ[forcesActive], angles='xy', scale=15, units='xy')

axs[1].axes.set_aspect('equal')
axs[1].set_xlabel('Y [in]')
axs[1].set_ylabel('Z [in]')
axs[1].set_title('Adjusted Shape (MAE = %.3f in)' %final_error.item())
first_legend = axs[1].legend(['Target shape', 'Adjusted shape'], loc='center')


# # # Add the legend manually to the current Axes.
axs[1].add_artist(first_legend)
# # # Create another legend for the second line.
axs[1].legend(['_nolegend_','_nolegend_','Unused actuators', 'Selected actuators', 'Force vectors'], loc='lower right', bbox_to_anchor=(1.55, 0.0))

axs[1].set_xlim(-170, 170)
axs[1].set_ylim(-170, 170)

plt.draw()